# Đo lường các nguồn tin tình báo về lừa đảo nhắm tới Việt Nam: làm lại từng bước và kiểm mọi con số

Notebook này làm lại toàn bộ phép đo của bài về các feed tình báo lừa đảo (ChongLuaDao, Tín Nhiệm Mạng, OpenPhish) và các danh sách benign, từ bộ dữ liệu
PhishVN đã công bố. Mọi phép tính nằm ngay trong notebook; notebook chỉ chạy, in kết quả, và so kết quả với bản kỳ vọng.

### Cách chạy

1. Tải thư mục chứa notebook này (gồm `feed_measurement_walkthrough.ipynb`, thư mục `expected/` và `requirements.txt`).
2. Cài thư viện: `pip install -r requirements.txt` (Python 3.12).
3. Mở notebook và chạy hết (Run All). Cả notebook chạy chưa tới một phút.

Lần chạy đầu, notebook tự tải gói `PhishVN_v3.1.0_open.zip` (khoảng 6 MB) từ bản 4 của bộ dữ liệu PhishVN trên Mendeley Data (DOI 10.17632/b97hxbxtpd.4),
kiểm SHA-256, rồi lấy ra `dataset_url.csv`. Thư mục `expected/` chứa ba file kết quả của bài để so, và một file đầu vào đã tính sẵn,
`first_seen_validation.csv` (xem Bước 9).

### Notebook kiểm gì

Mỗi bước tính một nhóm con số, ghi vào từ điển `STATS`, rồi so ngay từng con số với file kết quả `p7_results.json` của bài. Bước 11 so lại toàn bộ
`STATS` một lần nữa và kiểm không con số nào trong file bị bỏ sót. Hai bảng có file
CSV riêng (Bước 5 và 6) được so thêm từng byte. Phép dò DNS wildcard của bài không có trong notebook: đó là truy vấn DNS trực tiếp nên chạy lại sẽ ra kết quả
của hôm nay, không phải của lúc bài đo.

| Bước | Làm gì |
|---|---|
| 0 | thiết lập, tải dữ liệu |
| 1 | nạp corpus và gộp host theo `www.` |
| 2 | mức trùng `www.` giữa hai lớp |
| 3 | thành phần theo từng nguồn |
| 4 | chồng lấn giữa các feed phishing |
| 5 | chồng lấn trong cửa sổ thời gian chung |
| 6 | độ nhạy theo tầng bằng chứng |
| 7 | đuôi tên miền và tỷ lệ `.vn` |
| 8 | đặc trưng URL theo dòng và theo host |
| 9 | ngành bị mạo danh, và tên dịch vụ công |
| 10 | thời gian: phủ ngày, đỉnh, ngày nhập hàng loạt, ngày bị lệch |
| 11 | so toàn bộ kết quả và tổng hợp |

### Thuật ngữ

| Từ | Nghĩa |
|---|---|
| dòng (row) | một chỉ báo trong corpus (một URL kèm nhãn và nguồn) |
| host | tên miền đã viết thường và bỏ một nhãn `www.` ở đầu; `www.foo.vn` và `foo.vn` là một host |
| tầng bằng chứng (tier) | `bronze` (một nguồn cộng đồng), `silver`, `gold` (có thêm xác nhận) |
| ngành (scenario) | nhóm thương hiệu bị mạo danh: ngân hàng, thương mại điện tử, game, mạng xã hội, giao hàng, cơ quan nhà nước, viễn thông, thuế; `other` là chưa gắn |
| bootstrap theo cụm | lấy mẫu lại theo host (không theo dòng) để tính khoảng tin cậy 95%; cặp `www.` không bị tính như hai bằng chứng độc lập |

## Bước 0: Thiết lập và tải dữ liệu

Ô thứ hai định nghĩa các hàm kiểm: `check` ghi một phép so, `report` in số phép khớp của một bước, `check_file` so từng byte một file với bản kỳ vọng,
`check_stats` so từng con số trong `STATS` với `p7_results.json`, và `check_json` so cả một kết quả với một file JSON.

In [ ]:
import os, re, json, math, hashlib, zipfile, tempfile, filecmp, urllib.request, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

MENDELEY_ZIP = ("https://data.mendeley.com/public-files/datasets/b97hxbxtpd/files/"
                "d5724d4d-9c2b-46b7-810e-cf290c3bfd87/file_downloaded")
ZIP_SHA256 = "308351e9d0c0fca13a81f2f63524a7dee08ccba6c775848d125b3da5525d0ab5"
DATASET_SHA256 = "8e1c6325a82177014ac892a9aaf8c4d3cbbc986b47fd1ecac8899c865fb6eade"
EXPECTED_URL = "https://raw.githubusercontent.com/vuthainguyen1602/phishvn/master/notebooks/feed_measurement/expected"
EXPECTED_FILES = ["p7_results.json", "p7_common_window.csv", "p7_tier_sensitivity.csv", "first_seen_validation.csv"]

def fetch(url, dest):
    # Mendeley refuses Python's default user agent
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (PhishVN walkthrough notebook)"})
    Path(dest).write_bytes(urllib.request.urlopen(req, timeout=120).read())

HOME = Path.cwd()
WORK = HOME / "feed_work"
WORK.mkdir(exist_ok=True)
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
zip_path = WORK / "PhishVN_v3.1.0_open.zip"
if not zip_path.exists():
    fetch(MENDELEY_ZIP, zip_path)
assert sha(zip_path) == ZIP_SHA256, "the downloaded package is not the published one"
DATASET = WORK / "dataset_url.csv"
with zipfile.ZipFile(zip_path) as z:
    DATASET.write_bytes(z.read("data/dataset_url.csv"))
assert sha(DATASET) == DATASET_SHA256

EXPECTED = HOME / "expected"
if not all((EXPECTED / f).exists() for f in EXPECTED_FILES):
    EXPECTED = WORK / "expected"
    EXPECTED.mkdir(exist_ok=True)
    for f in EXPECTED_FILES:
        if not (EXPECTED / f).exists():
            fetch(f"{EXPECTED_URL}/{f}", EXPECTED / f)
FS_VALID = EXPECTED / "first_seen_validation.csv"
OUT = WORK / "results"
OUT.mkdir(exist_ok=True)
print("dataset:", DATASET, "| expected:", EXPECTED)

In [ ]:
CHECKS = []

def _same(a, b):
    return a == b or (isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b))

def check(step, name, mine, ref):
    ok = _same(mine, ref)
    CHECKS.append({"step": step, "check": name, "notebook": mine, "saved": ref, "match": bool(ok)})
    return ok

def report(step):
    rows_ = [c for c in CHECKS if c["step"] == step]
    bad = [c for c in rows_ if not c["match"]]
    print(f"[{step}] {len(rows_) - len(bad)}/{len(rows_)} checks match")
    for c in bad:
        print(f"    MISMATCH {c['check']}: notebook={c['notebook']!r}  saved={c['saved']!r}")

def check_file(step, produced, expected_name):
    same = filecmp.cmp(produced, EXPECTED / expected_name, shallow=False)
    check(step, f"{expected_name} byte-identical", same, True)

def _plain(o):
    return json.loads(json.dumps(o, default=lambda v: v.item() if hasattr(v, "item") else str(v)))

def flat(d, prefix=""):
    if isinstance(d, dict):
        for k, v in d.items():
            yield from flat(v, f"{prefix}.{k}" if prefix else str(k))
    elif isinstance(d, list) and d and all(isinstance(x, (dict, list)) for x in d):
        for i, v in enumerate(d):
            yield from flat(v, f"{prefix}[{i}]")
    else:
        yield prefix, d

SAVED = None  # p7_results.json, loaded in step 1

def check_stats(step, keys):
    # one check per number: every field of STATS[key] against the saved results
    for k in keys:
        new, ref = dict(flat(_plain(STATS[k]), k)), dict(flat(SAVED.get(k, "<missing>"), k))
        for f in sorted(set(new) | set(ref)):
            check(step, f, new.get(f, "<missing>"), ref.get(f, "<missing>"))
    report(step)

def check_json(step, label, mine, saved_file):
    new = dict(flat(_plain(mine)))
    ref = dict(flat(json.load(open(saved_file, encoding="utf-8"))))
    keys = sorted(set(new) | set(ref))
    diff = [k for k in keys if not _same(new.get(k, "<missing>"), ref.get(k, "<missing>"))]
    check(step, f"{label}: fields that differ (of {len(keys)})", len(diff), 0)
    for k in diff[:15]:
        print(f"    differs: {k}: saved={ref.get(k, '<missing>')!r} notebook={new.get(k, '<missing>')!r}")

## Bước 1: Nạp corpus và gộp host

Corpus là bảng `dataset_url.csv`: mỗi dòng một chỉ báo, kèm nhãn (`phishing`/`benign`), nguồn, tầng bằng chứng, ngành bị mạo danh, đuôi tên miền và ngày.

Đơn vị đếm quan trọng nhất của bài là host, không phải dòng. Corpus đã khử trùng theo URL nguyên văn, nhưng `foo.tld` và `www.foo.tld` là hai URL khác nhau
nên vẫn là hai dòng. Ở lớp phishing có gần 17.000 cặp như vậy, ở lớp benign gần như không có. Đếm theo dòng vì thế thổi phồng lớp phishing một cách lệch.
`host_key` viết thường và bỏ một nhãn `www.` ở đầu; `hosts(df)` giữ dòng đầu tiên của mỗi host theo thứ tự trong corpus (thứ tự cố định nên kết quả tất định).

In [ ]:
STATS = {}
SAVED = json.load(open(EXPECTED / "p7_results.json", encoding="utf-8"))
BOOT_SEED, BOOT_B, BULK_MULTIPLE = 20260819, 5000, 20
SECTORS = ["bank", "ecommerce", "gaming", "social", "delivery", "gov", "telecom", "tax"]

def host_key(s):
    return s.astype(str).str.strip().str.lower().str.replace(r"^www\.", "", regex=True)

def hosts(df, label=None):
    d = df if label is None else df[df.label == label]
    return d.drop_duplicates("host")

def parse_first_seen(s):
    # dd/mm/yyyy and ISO both occur; rows that parse under neither stay undated
    s = s.astype(str).str.strip().str.slice(0, 10)
    dt = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    return dt.fillna(pd.to_datetime(s, format="%Y-%m-%d", errors="coerce"))

df = pd.read_csv(DATASET, low_memory=False)
df["host"] = host_key(df["domain"])
df["is_www"] = df["domain"].astype(str).str.strip().str.lower().str.startswith("www.")
STATS["n_total"] = len(df)
STATS["n_phish"] = int((df.label == "phishing").sum())
STATS["n_benign"] = int((df.label == "benign").sum())
STATS["n_host_total"] = int(df.host.nunique())
STATS["n_host_phish"] = int(df.loc[df.label == "phishing", "host"].nunique())
STATS["n_host_benign"] = int(df.loc[df.label == "benign", "host"].nunique())
print({k: STATS[k] for k in ("n_total", "n_phish", "n_benign", "n_host_total", "n_host_phish", "n_host_benign")})
check_stats("1 corpus", ["n_total", "n_phish", "n_benign", "n_host_total", "n_host_phish", "n_host_benign"])

## Bước 2: Mức trùng `www.` giữa hai lớp

Bao nhiêu host của lớp phishing xuất hiện cả dạng có và không có `www.`, và tỷ lệ dòng mang `www.` ở mỗi lớp. Gộp cặp này làm lớp phishing mất một phần đáng kể
số dòng trong khi lớp benign gần như không đổi, nên mọi thống kê theo dòng của lớp phishing bị thổi phồng theo một hệ số riêng của nguồn cộng đồng chiếm đa số.

In [ ]:
ph, be = df[df.label == "phishing"], df[df.label == "benign"]
STATS["raw_hosts_phish"] = int(ph.domain.astype(str).str.lower().nunique())
STATS["www_pairs_phish"] = len(set(ph.loc[~ph.is_www, "host"]) & set(ph.loc[ph.is_www, "host"]))
STATS["www_pairs_benign"] = len(set(be.loc[~be.is_www, "host"]) & set(be.loc[be.is_www, "host"]))
STATS["www_rows_phish"], STATS["www_rows_benign"] = int(ph.is_www.sum()), int(be.is_www.sum())
print(f"phishing: {len(ph):,} rows, {STATS['raw_hosts_phish']:,} host strings, {STATS['n_host_phish']:,} hosts; "
      f"{STATS['www_pairs_phish']:,} hosts listed both with and without www.")
print(f"rows carrying www.: phishing {100 * STATS['www_rows_phish'] / len(ph):.1f}%, benign {100 * STATS['www_rows_benign'] / len(be):.1f}%")
print(f"collapsing the pairs removes {100 * (1 - STATS['n_host_phish'] / len(ph)):.1f}% of phishing rows "
      f"and {100 * (1 - STATS['n_host_benign'] / len(be)):.1f}% of benign rows")
check_stats("2 www pairs", ["raw_hosts_phish", "www_pairs_phish", "www_pairs_benign", "www_rows_phish", "www_rows_benign"])

## Bước 3: Thành phần theo từng nguồn

Với mỗi nguồn: số dòng, số host, tỷ trọng trong corpus, tỷ lệ phishing, và số dòng ở mỗi tầng bằng chứng. Cột host không cộng được qua các nguồn vì một host có thể
nằm ở nhiều nguồn.

In [ ]:
rows = []
for s in df.source.value_counts().index.tolist():
    sub = df[df.source == s]
    tm = sub.tier.value_counts()
    rows.append((s, len(sub), int(sub.host.nunique()), 100 * len(sub) / len(df), 100 * (sub.label == "phishing").mean(),
                 int(tm.get("bronze", 0)), int(tm.get("silver", 0)), int(tm.get("gold", 0))))
STATS["feed_composition"] = [{"source": s, "rows": n, "hosts": nh, "pct_corpus": round(float(pct), 4),
                              "pct_phish": round(float(pph), 4), "bronze": b, "silver": si, "gold": g}
                             for (s, n, nh, pct, pph, b, si, g) in rows]
STATS["tier_totals"] = {t: int((df.tier == t).sum()) for t in ("bronze", "silver", "gold")}
display(pd.DataFrame(STATS["feed_composition"]))
print("tier totals:", STATS["tier_totals"])
check_stats("3 composition", ["feed_composition", "tier_totals"])

## Bước 4: Chồng lấn giữa các feed phishing

Giá trị thật của việc gộp nhiều feed là những host chỉ một feed có. Với mỗi feed phishing: số dòng và tỷ lệ trong lớp phishing, số host và tỷ lệ trong hợp của
cả ba feed, số host chỉ feed đó có. Kèm số host chung của từng cặp feed.

In [ ]:
sets = {s: set(g.host) for s, g in ph.groupby("source")}
union = set().union(*sets.values())
order = sorted(sets, key=lambda s: -len(sets[s]))
rows = []
for s in order:
    others = set().union(*[sets[o] for o in sets if o != s])
    uniq = sets[s] - others
    nrows = int((ph.source == s).sum())
    rows.append((s, nrows, 100 * nrows / len(ph), len(sets[s]), 100 * len(sets[s]) / len(union), len(uniq),
                 100 * len(uniq) / len(sets[s])))
STATS["feed_rows_vs_hosts"] = {r[0]: (r[2], r[4]) for r in rows}
STATS["feed_unique_hosts"] = {r[0]: (r[5], r[6]) for r in rows}
STATS["feed_pair_overlap"] = {f"{a}|{b}": len(sets[a] & sets[b]) for i, a in enumerate(order) for b in order[i + 1:]}
display(pd.DataFrame(rows, columns=["feed", "rows", "% of rows", "hosts", "% of union", "unique hosts", "% of feed"]).round(2))
print("shared hosts per pair:", STATS["feed_pair_overlap"])
check_stats("4 feed overlap", ["feed_rows_vs_hosts", "feed_unique_hosts", "feed_pair_overlap"])

## Bước 5: Chồng lấn trong cửa sổ thời gian chung

Hai feed có ngày theo từng chỉ báo (ChongLuaDao và Tín Nhiệm Mạng) hoạt động ở những khoảng thời gian khác nhau, nên so chồng lấn trên toàn bộ dữ liệu có thể chỉ
phản ánh việc chúng không cùng thời. Bước này cắt cả hai về giao của hai khoảng thời gian có ngày, rồi đo lại: số host mỗi feed, số host chung, hệ số Jaccard
(chung chia hợp) và tỷ lệ chung trên feed nhỏ hơn. OpenPhish bị loại vì không dòng nào trong 296 dòng của nó có ngày; gán ngày thu thập của bản chụp làm ngày sự
kiện sẽ biến một bản chụp thành một feed theo thời gian.

In [ ]:
php = ph.copy()
php["date"] = parse_first_seen(php["collected_at"])
SRC2 = ("chongluadao", "tinnhiemmang")
dated = php[php.source.isin(SRC2) & php.date.notna()]
spans = dated.groupby("source").date.agg(["min", "max"])
start, end = spans["min"].max(), spans["max"].min()

def window_row(condition, d, lo, hi):
    s_ = {s: set(d[d.source == s].host) for s in SRC2}
    inter, uni = s_[SRC2[0]] & s_[SRC2[1]], s_[SRC2[0]] | s_[SRC2[1]]
    smaller = min(len(s_[SRC2[0]]), len(s_[SRC2[1]]))
    return {"condition": condition, "start": lo, "end": hi, "chongluadao_hosts": len(s_["chongluadao"]),
            "tinnhiemmang_hosts": len(s_["tinnhiemmang"]), "overlap_hosts": len(inter), "union_hosts": len(uni),
            "jaccard": len(inter) / len(uni) if uni else np.nan, "share_of_smaller": len(inter) / smaller if smaller else np.nan,
            "excluded_source": "openphish", "excluded_reason": "no per-indicator first-seen dates (0/296 dated)"}

cw = pd.DataFrame([window_row("all_retrievable", php[php.source.isin(SRC2)], "source-specific", "source-specific"),
                   window_row("common_dated_window", dated[dated.date.between(start, end)], start.date(), end.date())])
cw.to_csv(OUT / "p7_common_window.csv", index=False, float_format="%.6f")
STATS["common_window"] = cw.to_dict("records")
print(f"common dated window: {start.date()} to {end.date()}")
display(cw.drop(columns=["excluded_source", "excluded_reason"]))
check_file("5 common window", OUT / "p7_common_window.csv", "p7_common_window.csv")
check_stats("5 common window", ["common_window"])

## Bước 6: Độ nhạy theo tầng bằng chứng

Tính lại các con số chính riêng cho từng tầng (`bronze`, `silver`, `gold`, và `silver` + `gold` gộp), theo host: tỷ lệ `.vn`, tỷ lệ đuôi tên miền đáng ngờ, tỷ lệ đã
gắn ngành, tỷ lệ ngân hàng và thương mại điện tử, và hiệu thương mại điện tử trừ ngân hàng kèm khoảng tin cậy bootstrap (5.000 lần, seed 20260819). Tầng và nguồn
gần như trùng nhau trong corpus này, nên bảng chỉ mô tả hướng nào còn đứng khi giới hạn bằng chứng, không ước lượng được tác động riêng của việc có xác nhận.

`cluster_bootstrap` lấy mẫu lại các cụm (host) có hoàn lại và trả về tổng từng cột cho mỗi lần lấy; cột đầu là số dòng của cụm, nên một tỷ lệ là cột j chia cột 0.
Hàm chia nhỏ theo khối để giới hạn bộ nhớ.

In [ ]:
def boot_ci(draws):
    lo, hi = np.percentile(draws, [2.5, 97.5])
    return float(lo), float(hi)

def cluster_bootstrap(counts, seed=BOOT_SEED, B=BOOT_B):
    rng = np.random.default_rng(seed)
    K = counts.shape[0]
    out = np.empty((B, counts.shape[1]))
    step = max(1, int(5e6 // max(K, 1)))
    for lo in range(0, B, step):
        hi = min(B, lo + step)
        idx = rng.integers(0, K, size=(hi - lo, K))
        out[lo:hi] = counts[idx].sum(axis=1)
    return out

per_tier = ph.drop_duplicates(["tier", "host"])
groups = [("bronze", per_tier[per_tier.tier == "bronze"]), ("silver", per_tier[per_tier.tier == "silver"]),
          ("gold", per_tier[per_tier.tier == "gold"]),
          ("verified", ph[ph.tier.isin(["silver", "gold"])].drop_duplicates("host"))]
rows = []
for name, g in groups:
    one = pd.DataFrame({"n": 1.0, "ecom": g.scenario.eq("ecommerce").astype(float),
                        "bank": g.scenario.eq("bank").astype(float)}).to_numpy(float)
    dr = cluster_bootstrap(one)
    lo, hi = boot_ci(100 * (dr[:, 1] - dr[:, 2]) / dr[:, 0])
    rows.append({"tier": name, "n_hosts": len(g), "vn_pct": 100 * g.tld.astype(str).eq("vn").mean(),
                 "suspicious_tld_pct": 100 * pd.to_numeric(g.suspicious_tld).mean(),
                 "sector_tagged_pct": 100 * g.scenario.ne("other").mean(), "bank_pct": 100 * g.scenario.eq("bank").mean(),
                 "ecommerce_pct": 100 * g.scenario.eq("ecommerce").mean(),
                 "ecom_minus_bank_pp": 100 * (g.scenario.eq("ecommerce").mean() - g.scenario.eq("bank").mean()),
                 "ecom_minus_bank_ci_lo": lo, "ecom_minus_bank_ci_hi": hi,
                 "sources": ";".join(sorted(g.source.astype(str).unique()))})
tiers = pd.DataFrame(rows)
tiers.to_csv(OUT / "p7_tier_sensitivity.csv", index=False, float_format="%.6f")
b_, v_ = tiers.set_index("tier").loc["bronze"], tiers.set_index("tier").loc["verified"]
STATS["tier_sensitivity"] = tiers.to_dict("records")
STATS["bronze_order_unresolved"] = bool(b_.ecom_minus_bank_ci_lo <= 0 <= b_.ecom_minus_bank_ci_hi)
display(tiers.drop(columns=["sources"]).round(2))
print("bronze tier leaves e-commerce vs banking unresolved (CI contains 0):", STATS["bronze_order_unresolved"])
check_file("6 tier sensitivity", OUT / "p7_tier_sensitivity.csv", "p7_tier_sensitivity.csv")
check_stats("6 tier sensitivity", ["tier_sensitivity", "bronze_order_unresolved"])

## Bước 7: Đuôi tên miền và tỷ lệ `.vn`

Tỷ lệ `.vn` của lớp benign (khoảng 61,6% theo dòng) không phải là một thống kê về các trang hợp lệ của Việt Nam: nó là trung bình có trọng số của bốn danh sách
benign, mà tỷ lệ `.vn` riêng của chúng chạy từ 0,3% tới 100%, nên con số đổi theo số dòng lấy từ mỗi danh sách. Bước này in tỷ lệ `.vn` theo dòng và theo host
cho cả hai lớp, theo từng nguồn benign, tỷ lệ đuôi đáng ngờ, và bảng 12 đuôi tên miền nhiều host nhất (phishing theo dòng và theo host, benign theo host).

In [ ]:
STATS["vn_share_phish"] = round(float(100 * (ph.tld == "vn").mean()), 1)
STATS["vn_share_benign"] = round(float(100 * (be.tld == "vn").mean()), 1)
STATS["n_tld"] = int(df.tld.nunique())
STATS["top_tlds"] = df.tld.value_counts().head(10).index.tolist()

per = (be.groupby("source").apply(lambda g: pd.Series({"n": len(g), "vn": 100 * (g.tld == "vn").mean()}), include_groups=False)
         .sort_values("n", ascending=False))
beh, phh = hosts(df, "benign"), hosts(df, "phishing")
STATS["vn_share_benign_host"] = round(float(100 * (beh.tld == "vn").mean()), 1)
STATS["vn_share_phish_host"] = round(float(100 * (phh.tld == "vn").mean()), 1)
STATS["vn_per_source"] = {s: round(float(r.vn), 1) for s, r in per.iterrows()}
STATS["susp_tld_phish_host"] = round(float(100 * phh.suspicious_tld.mean()), 1)
STATS["susp_tld_benign_host"] = round(float(100 * beh.suspicious_tld.mean()), 1)

top = hosts(df).tld.value_counts().head(12).index.tolist()
ph_c, uph_c, ube_c = ph.tld.value_counts(), phh.tld.value_counts(), beh.tld.value_counts()
tld_rows = []
for t in top:
    up, ub = int(uph_c.get(t, 0)), int(ube_c.get(t, 0))
    tld_rows.append({"tld": t, "phish_rows": int(ph_c.get(t, 0)), "phish_hosts": up, "benign_hosts": ub,
                     "phish_share_hosts": round(float(100 * up / (up + ub) if up + ub else 0), 4)})
STATS["tld_table"] = tld_rows
STATS["susp_tld_phish"] = round(float(100 * ph.suspicious_tld.mean()), 1)
STATS["susp_tld_benign"] = round(float(100 * be.suspicious_tld.mean()), 1)

print(f".vn share by row: phishing {STATS['vn_share_phish']}%, benign {STATS['vn_share_benign']}%; "
      f"by host: phishing {STATS['vn_share_phish_host']}%, benign {STATS['vn_share_benign_host']}%")
print("benign .vn share per source:", STATS["vn_per_source"])
display(pd.DataFrame(tld_rows))
check_stats("7 tld", ["vn_share_phish", "vn_share_benign", "n_tld", "top_tlds", "vn_share_benign_host", "vn_share_phish_host", "vn_per_source", "susp_tld_phish_host", "susp_tld_benign_host", "tld_table", "susp_tld_phish", "susp_tld_benign"])

## Bước 8: Đặc trưng URL theo dòng và theo host

Độ dài chuỗi URL (với khoảng 95,6% corpus, URL chính là host) và số tên miền con, tính trung vị, trung bình và khoảng tứ phân vị cho từng lớp, cả theo dòng và
theo host. Theo dòng, lớp phishing có vẻ nhiều tên miền con hơn; nhưng đó hoàn toàn là cặp `www.`, mỗi bản `www.` thêm một tên miền con mà bản không `www.` không
có. Gộp cặp đi thì thứ tự đảo lại: chính các trang hợp lệ của Việt Nam có chồng tên miền con sâu hơn.

In [ ]:
STATS["url_is_host_pct"] = round(float(100 * (df.url.astype(str).str.lower() == df.domain.astype(str).str.lower()).mean()), 1)
for col in ("url_len", "num_subdomains"):
    for tag, pair in (("row", (ph, be)), ("host", (phh, beh))):
        for cls, sub in zip(("phish", "benign"), pair):
            v = sub[col].dropna()
            STATS[f"{col}_{tag}_med_{cls}"] = float(v.median())
            STATS[f"{col}_{tag}_mean_{cls}"] = round(float(v.mean()), 3)
            STATS[f"{col}_{tag}_iqr_{cls}"] = [float(v.quantile(.25)), float(v.quantile(.75))]
for k, v in STATS.items():
    if k.startswith(("url_len_", "num_subdomains_")):
        print(f"  {k:32s} {v}")
check_stats("8 url features", ["url_is_host_pct"] + [k for k in STATS if k.startswith(("url_len_", "num_subdomains_"))])

## Bước 9: Ngành bị mạo danh, và tên dịch vụ công

Hiệu ngân hàng trừ thương mại điện tử, theo dòng và theo host, kèm khoảng tin cậy bootstrap theo cụm host. Theo dòng thì các bản `www.` bị tính như bằng chứng độc
lập; theo host thì không. Hai cách cho kết quả khác nhau, và chính sự khác nhau đó là phát hiện. Tiếp theo là tỷ lệ host của từng ngành kèm khoảng tin cậy, số host
đã gắn ngành, và tỷ lệ chưa gắn.

Cuối cùng là tên dịch vụ công: tìm ba token cố định (`dichvucong` cổng dịch vụ công quốc gia, `vssid` bảo hiểm xã hội, `gplx` giấy phép lái xe) trong các host phishing.
Token phải bắt đầu một nhãn hoặc đứng sau một ký tự không phải chữ cái, trừ token từ 9 ký tự trở lên. Danh sách token được cố định trước, không lấy từ corpus, để
phép tìm không thể được chỉnh cho tới khi tìm ra thứ gì đó.

In [ ]:
g = ph.groupby("host").scenario
cnt = pd.DataFrame({"n": g.size(), "bank": g.apply(lambda s: int((s == "bank").sum())),
                    "ecom": g.apply(lambda s: int((s == "ecommerce").sum()))})
draws = cluster_bootstrap(cnt[["n", "bank", "ecom"]].to_numpy(float))
lo_r, hi_r = boot_ci(100 * (draws[:, 1] - draws[:, 2]) / draws[:, 0])
obs_r = 100 * ((ph.scenario == "bank").mean() - (ph.scenario == "ecommerce").mean())
u = phh
one = pd.DataFrame({"n": 1.0, "bank": (u.scenario == "bank").astype(float), "ecom": (u.scenario == "ecommerce").astype(float)})
dh = cluster_bootstrap(one.to_numpy(float))
lo_h, hi_h = boot_ci(100 * (dh[:, 1] - dh[:, 2]) / dh[:, 0])
obs_h = 100 * ((u.scenario == "bank").mean() - (u.scenario == "ecommerce").mean())
STATS["bank_minus_ecom_row"] = (round(obs_r, 3), round(lo_r, 3), round(hi_r, 3))
STATS["bank_minus_ecom_host"] = (round(obs_h, 3), round(lo_h, 3), round(hi_h, 3))
ci = {}
for s in SECTORS:
    dr = cluster_bootstrap(pd.DataFrame({"n": 1.0, "s": (u.scenario == s).astype(float)}).to_numpy(float))
    ci[s] = (100 * float((u.scenario == s).mean()), *boot_ci(100 * dr[:, 1] / dr[:, 0]))
STATS["sector_host_ci"] = {k: tuple(round(x, 2) for x in v) for k, v in ci.items()}

hcounts = u.scenario.value_counts()
h_other = int(hcounts.get("other", 0))
STATS["sector_host_counts"] = {s: int(hcounts.get(s, 0)) for s in SECTORS}
STATS["n_tagged_hosts"] = len(u) - h_other
STATS["h_tagged_pct"] = round(100 * (len(u) - h_other) / len(u), 2)
STATS["h_other_pct"] = round(100 * h_other / len(u), 2)
print("bank minus e-commerce (pp, 95% CI): by row", STATS["bank_minus_ecom_row"], "| by host", STATS["bank_minus_ecom_host"])
display(pd.DataFrame(STATS["sector_host_ci"], index=["host share %", "CI low", "CI high"]).T.join(
    pd.Series(STATS["sector_host_counts"], name="hosts")))
print(f"hosts tagged with a sector: {STATS['n_tagged_hosts']:,} ({STATS['h_tagged_pct']}%), untagged {STATS['h_other_pct']}%")

def token_at_boundary(domain, tok):
    for label in re.split(r"[.\-_]", domain.lower()):
        i = label.find(tok)
        while i != -1:
            if i == 0 or not label[i - 1].isalpha() or len(tok) >= 9:
                return True
            i = label.find(tok, i + 1)
    return False

gov = {}
for tok in ("dichvucong", "vssid", "gplx"):
    m = u[u.host.map(lambda h, t=tok: token_at_boundary(h, t))]
    gov[tok] = (len(m), int((m.scenario == "gov").sum()))
STATS["gov_token_hosts"] = gov
print("public-service tokens (hosts matched, of which tagged gov):", gov)
check_stats("9 sectors", ["bank_minus_ecom_row", "bank_minus_ecom_host", "sector_host_ci", "sector_host_counts", "n_tagged_hosts", "h_tagged_pct", "h_other_pct", "gov_token_hosts"])

## Bước 10: Thời gian

Phần thời gian chỉ dùng các dòng có ngày first-seen. Bước này in:
- độ phủ ngày của corpus, ngày sớm nhất và muộn nhất, số dòng phishing có ngày, và tháng đỉnh của lớp phishing (chỉ tính phishing, vì ngày của lớp benign là ngày
  chứng nhận của cơ quan đăng ký chứ không phải ngày phát hiện);
- ngày cuối cùng mỗi nguồn còn đăng chỉ báo. Phần đuôi mỏng dần sau 2023 là do cả hai feed phishing ngừng đăng, không phải do bài ngừng thu thập (danh sách benign
  vẫn chạy tới 2026);
- ngày nhập hàng loạt của ChongLuaDao: những ngày có số bản ghi từ 20 lần trung vị theo ngày trở lên, và phần trăm khối lượng nằm ở các ngày đó. Ngày của ChongLuaDao
  được khôi phục từ mã định danh trong cơ sở dữ liệu, tức là ngày nhập vào cơ sở dữ liệu chứ không phải ngày phát hiện;
- với các tên miền có ở cả ChongLuaDao và Tín Nhiệm Mạng (ngày của Tín Nhiệm Mạng là ngày được xác nhận), độ lệch ngày tuyệt đối trung vị, tách theo ngày nhập hàng
  loạt và ngày thường. Ở ngày nhập hàng loạt độ lệch lớn hơn hẳn: đó là dấu hiệu của việc nhập dồn một kho tồn, không phải một đợt phát hiện thật. Phép so này dùng
  `first_seen_validation.csv` đi kèm trong `expected/`; file đó được tính bằng cách đối chiếu với dữ liệu thô của ChongLuaDao, vốn không công bố;
- số host có ít nhất một dòng có ngày, cho lớp phishing và cả corpus. Việc gộp `www.` làm con số này đổi nhiều hơn mọi con số khác của bài.

In [ ]:
dt = parse_first_seen(df["collected_at"])
dated_all = dt.dropna()
STATS["n_dated"] = int(len(dated_all))
STATS["dated_coverage_pct"] = round(float(100 * len(dated_all) / len(df)), 1)
STATS["date_min"], STATS["date_max"] = str(dated_all.min().date()), str(dated_all.max().date())
ph_dated = dt[df["label"].to_numpy() == "phishing"].dropna()
STATS["n_dated_phishing"] = int(len(ph_dated))
monthly = ph_dated.dt.to_period("M").value_counts().sort_index()
monthly = monthly.reindex(pd.period_range(monthly.index.min(), monthly.index.max(), freq="M"), fill_value=0)
STATS["peak_month"], STATS["peak_month_n"] = str(monthly.idxmax()), int(monthly.max())

lanes = (pd.DataFrame({"t": dt, "source": df["source"].astype(str), "label": df["label"].astype(str)}).dropna(subset=["t"])
           .groupby(["source", "label"]).agg(n=("t", "size"), first=("t", "min"), last=("t", "max")).reset_index().sort_values("last"))
STATS["feed_last_phishing"] = {r.source: str(r.last.date()) for r in lanes[lanes.label == "phishing"].itertuples()}
STATS["feed_last_benign"] = {r.source: str(r.last.date()) for r in lanes[lanes.label == "benign"].itertuples()}
print(f"dated rows {STATS['n_dated']:,} ({STATS['dated_coverage_pct']}%), {STATS['date_min']} to {STATS['date_max']}; "
      f"peak phishing month {STATS['peak_month']} ({STATS['peak_month_n']:,})")
display(lanes)

cld = df[(df.label == "phishing") & (df.source == "chongluadao")].copy()
cld["t"] = parse_first_seen(cld["collected_at"])
daily = cld.dropna(subset=["t"]).t.dt.floor("D").value_counts().sort_index()
thr = BULK_MULTIPLE * float(daily.median())
bulk = set(daily[daily >= thr].index)
tot = int(daily.sum())
STATS["bulk_days"] = [f"{d:%Y-%m-%d}" for d in sorted(bulk)]
STATS["bulk_mass_pct"] = round(100 * int(daily[daily.index.isin(bulk)].sum()) / tot, 1)
STATS["top20_day_mass_pct"] = round(100 * int(daily.sort_values(ascending=False).head(20).sum()) / tot, 1)
STATS["n_dated_days"] = int(len(daily))
STATS["peak_day"] = (f"{daily.idxmax():%Y-%m-%d}", int(daily.max()))
v = pd.read_csv(FS_VALID)
o = v[v.check == "ncsc_overlap"].copy()
o["bulk"] = pd.to_datetime(o.cld_date).isin(bulk)
a_, e_ = o.loc[o.bulk, "delta_days"].abs(), o.loc[~o.bulk, "delta_days"].abs()
STATS["ncsc_delta_bulk"] = (int(len(a_)), float(a_.median()))
STATS["ncsc_delta_other"] = (int(len(e_)), float(e_.median()))
print(f"bulk-ingest days (>= {BULK_MULTIPLE} x median daily volume): {len(bulk)}, carrying {STATS['bulk_mass_pct']}% of dated records")
print(f"median |date difference| against the attested national feed: bulk days {STATS['ncsc_delta_bulk']}, other days {STATS['ncsc_delta_other']}")

dph = ph.assign(d=parse_first_seen(ph["collected_at"]).notna()).groupby("host").d.any()
d_all = df.assign(d=dt.notna()).groupby("host").d.any()
STATS["dated_hosts_phish"] = (int(dph.sum()), round(float(100 * dph.mean()), 1))
STATS["dated_hosts_corpus"] = (int(d_all.sum()), round(float(100 * d_all.mean()), 1))
print("hosts with at least one dated row: phishing", STATS["dated_hosts_phish"], "| corpus", STATS["dated_hosts_corpus"])
check_stats("10 time", ["n_dated", "dated_coverage_pct", "date_min", "date_max", "n_dated_phishing", "peak_month", "peak_month_n", "feed_last_phishing", "feed_last_benign", "bulk_days", "bulk_mass_pct", "top20_day_mass_pct", "n_dated_days", "peak_day", "ncsc_delta_bulk", "ncsc_delta_other", "dated_hosts_phish", "dated_hosts_corpus"])

## Bước 11: So toàn bộ kết quả và tổng hợp

Ô dưới so toàn bộ `STATS` (mọi con số tính ở Bước 1 tới 10) với `p7_results.json` thêm một lần, kiểm hai tập khoá trùng nhau (không con số nào trong file
bị bỏ sót, notebook cũng không tính thừa), rồi gom mọi phép kiểm theo bước.

In [ ]:
check_json("11 all results", "p7_results.json", STATS, EXPECTED / "p7_results.json")
check("11 all results", "keys in p7_results.json not computed here", sorted(set(SAVED) - set(STATS)), [])
check("11 all results", "keys computed here but absent from p7_results.json", sorted(set(STATS) - set(SAVED)), [])
report("11 all results")
tbl = pd.DataFrame(CHECKS)
display(tbl.groupby("step", sort=False)["match"].agg(checks="size", matched="sum"))
print(f"total: {int(tbl['match'].sum())}/{len(tbl)} checks match")